# Kestrel V3, step 5: re-SFT, chat eval, Roost, export

One GPU runtime (A100, H100 or G4), one `Run all`, every stage resumes from Drive.

| stage | what | cost |
|---|---|---|
| A | step 1's SFT mix with a larger refusal slice (6%, was 2%) and more contrast questions (3%, was 1%) | ~0.5 unit |
| B | SFT from step 4's model (`runs/pretrain_v3`), 2 epochs, step 1's LRs | ~5 units |
| C | the fixed chat eval on the new model, recounted next to V1, step 1 and step 2 | ~1 unit |
| D | Roost: re-teach the 7 self-facts (V2's `row-self-x2` setting, same gate) | ~1 unit |
| E | export f32/f16/q8_0 `.gguf`, check the C# engine against PyTorch, save to Drive `release_v3/` | ~0.5 unit |

Disconnect and delete the runtime when `STAGE_E_DONE` prints.


In [ ]:
# 0. Which runtime?
import os, json, shutil, glob, subprocess, time
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '|', GPU or 'CPU runtime')
if not GPU or torch.cuda.get_device_properties(0).total_memory < 39e9:
    raise SystemExit(f'Got {GPU}. Use an A100, H100 or G4 runtime.')


In [ ]:
# 1. Drive. The first mount right after the permission prompt often fails with 'mount failed'.
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError as e:
    print('first mount failed, retrying:', e)
    drive.mount('/content/drive', force_remount=True)
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
SFT5 = f'{KD}/datasets/sft_v5'
RUN = f'{KD}/runs/sft_v5'
PRE = f'{KD}/runs/pretrain_v3/ckpt.pt'
REL = f'{KD}/release_v3'
for d in (f'{KD}/evals', REL):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(PRE), PRE
print('SFT set built:', os.path.exists(f'{SFT5}/manifest.json'), '| SFT started:', os.path.exists(f'{RUN}/ckpt.pt'))


In [ ]:
# 2. Code (public repo)
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers datasets


In [ ]:
# A. The SFT set: step 1's mix, refusal slice 2% -> 6%, contrast 1% -> 3%. Built on this runtime
#    (a separate CPU runtime would need another Drive permission click). manifest.json is
#    written last, so its presence on Drive means the set is complete.
LOCAL = '/content/data_sft_v5'
if not os.path.exists(f'{SFT5}/manifest.json'):
    !python -u -m scripts.prepare_sft_v3 --examples 25000 --seq 4096 --harmful-share 0.06 --contrast-share 0.03 --out {LOCAL}
    assert os.path.exists(f'{LOCAL}/manifest.json'), 'build failed - see the log above'
    os.makedirs(SFT5, exist_ok=True)
    for f in ['train_tokens.bin', 'train_mask.bin', 'val_tokens.bin', 'val_mask.bin', 'samples.jsonl', 'manifest.json']:
        shutil.copy(f'{LOCAL}/{f}', f'{SFT5}/{f}')
m = json.load(open(f'{SFT5}/manifest.json'))
print(json.dumps({k: m.get(k) for k in ('examples', 'tokens', 'per_source', 'shortfall')}, indent=1))
print('STAGE_A_DONE')


In [ ]:
# B. SFT from step 4's model with fresh optimizers, step 1's recipe: same LRs, 196,608
#    tokens/step, about 2 epochs. --schedule-start keeps one LR curve across a resume.
os.makedirs(LOCAL, exist_ok=True)
for f in glob.glob(f'{SFT5}/*'):
    out = f'{LOCAL}/{os.path.basename(f)}'
    if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
        shutil.copy(f, out)
START = torch.load(PRE, map_location='cpu', weights_only=False)['step']
n_steps = round(2 * m['tokens']['train'] / 196608)
final = START + n_steps
print(f"{m['tokens']['train']/1e6:.1f}M train tokens -> {n_steps} steps, {START} -> {final}")
def finished():
    p = f'{RUN}/status.json'
    return os.path.exists(p) and json.load(open(p)).get('state') == 'done'
if not finished():
    !python -u -m kestrel.train --preset nano --sft-dir {LOCAL} --init-from {PRE} --schedule-start {START} \
        --bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 16 --accum 3 \
        --steps {final} --warmup 15 --decay-frac 0.5 --muon-lr 0.004 --adamw-lr 0.0006 \
        --rope-scale 4 --out {RUN} --log-every 5 --eval-every 25 --eval-iters 10 \
        --ckpt-every 50 --ckpt-secs 1800
assert finished(), 'SFT did not finish - see the log above. Run all again to resume.'
print('STAGE_B_DONE')


In [ ]:
# C. The same chat eval as steps 1 and 2, then a recount of every run with the fixed counters.
OUT = f'{KD}/evals/chat_eval_v3.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
if 'step5' not in done:
    !python -u -m scripts.eval_chat --ckpt {RUN}/ckpt.pt --label step5 --out {OUT}
!python -u -m scripts.eval_chat --rescore --out {OUT} > /dev/null
for k, r in json.load(open(OUT)).items():
    print(f'{k:>6}', json.dumps(r['summary']))
print('STAGE_C_DONE')


In [ ]:
# D. Roost: steps 4 and 5 changed the memory slots, so re-teach the 7 self-facts with the
#    setting that built V2 (one fact per row, 1,200 steps). It ships only if it passes its own
#    gate (recall >= 60%, regression <= 2%); otherwise the SFT model ships as it is.
A = '/content/assets'; os.makedirs(A, exist_ok=True)
if not os.path.isdir(f'{A}/roost-data'):
    !wget -q -O {A}/roost-data.zip https://github.com/YoMosa2009/Kestrel/releases/download/v1.0.0/roost-data.zip
    !unzip -q -o {A}/roost-data.zip -d {A}/roost-data
ROOST = '/content/v3roost'
if not os.path.exists(f'{REL}/roost-results.json'):
    !python -u -m scripts.roost_v2 --ckpt {RUN}/ckpt.pt --replay-dir {A}/roost-data --out {ROOST} --skip-benchmark --v2-setting row-self-x2 2>&1 | tee /content/v3roost.log
    shutil.copy(f'{ROOST}/results.json', f'{REL}/roost-results.json')
    if os.path.exists(f'{ROOST}/kestrel-nano-v2-weights.pt'):
        shutil.copy(f'{ROOST}/kestrel-nano-v2-weights.pt', f'{REL}/roost-weights.pt')
R = json.load(open(f'{REL}/roost-results.json'))
c = R.get('v2_consolidation', {})
print('self-facts recall', c.get('recall_before'), '->', c.get('recall_after'),
      '| regression', c.get('regression_pct'), '% | built:', R['v2_built'])
print('before:', R['v1_chat'])
print('after: ', R.get('v2_chat'))
W = f'{REL}/roost-weights.pt' if R['v2_built'] else f'{RUN}/ckpt.pt'
print('shipping', W)
print('STAGE_D_DONE')


In [ ]:
# E. Export, then check the native C# engine against PyTorch logits.
G = '/content/gguf'; os.makedirs(G, exist_ok=True)
NAME = '--name "Kestrel-Nano V3" --version V3'
!python -m scripts.export_gguf --ckpt {W} --out {G}/kestrel-nano-v3-f32.gguf --dtype f32 {NAME} --fixture {G}/parity-v3.json --save-pt {G}/kestrel-nano-v3-weights.pt
!python -m scripts.export_gguf --ckpt {W} --out {G}/kestrel-nano-v3-f16.gguf --dtype f16 {NAME}
!python -m scripts.export_gguf --ckpt {W} --out {G}/kestrel-nano-v3-q8_0.gguf --dtype q8_0 {NAME}
DOTNET = os.path.expanduser('~/.dotnet/dotnet')
if not os.path.exists(DOTNET):
    !curl -sSL https://dot.net/v1/dotnet-install.sh | bash -s -- --channel 8.0 > /dev/null
for q in ('f32', 'f16', 'q8_0'):
    !DOTNET_CLI_TELEMETRY_OPTOUT=1 DOTNET_NOLOGO=1 {DOTNET} run -c Release --project app/Kestrel.Cli -- parity {G}/kestrel-nano-v3-{q}.gguf {G}/parity-v3.json | tee {G}/parity-v3-{q}.txt
!md5sum {G}/*.gguf | tee {G}/md5.txt
for f in os.listdir(G):
    if not f.endswith('f32.gguf'):
        shutil.copy(f'{G}/{f}', f'{REL}/{f}')
print(sorted(os.listdir(REL)))
print('STAGE_E_DONE')
